# L5: Criação de conteúdo em escala

## Initial Imports

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew
from crewai import LLM

## Creating Structured Output

In [2]:
from pydantic import BaseModel, Field
from typing import List

class SocialMediaPost(BaseModel):
    platform: str = Field(..., description="A plataforma de mídia social onde a postagem será publicada (por exemplo, Twitter, LinkedIn).")
    content: str = Field(..., description="O conteúdo da postagem nas redes sociais, incluindo hashtags ou menções.")

class ContentOutput(BaseModel):
    article: str = Field(..., description="O artigo, formatado em markdown.")
    social_media_posts: List[SocialMediaPost] = Field(..., description="Uma lista de postagens nas redes sociais relacionadas ao artigo.")

## Loading Tasks and Agents YAML files

In [3]:
# Define file paths for YAML configurations
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Importing CrewAI Tools

In [4]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

## Setup Multi LLM models

In [5]:
os.environ['OPENAI_MODEL_NAME'] = 'gpt-4o-mini'
groq_llm = "llama-3.3-70b-versatile"

In [6]:
grop_llm = LLM(model="llama-3.3-70b-versatile")
llm_model = LLM(model="gpt-4o-mini")

## Creating Crew, Agents, and Tasks

In [10]:
from crewai.llms import LiteLLM  # ajuste o import conforme sua versão/configuração

# Instancie o LLM com o provedor desejado e, se necessário, as credenciais (por exemplo, api_key)
llm_model = LiteLLM(provider="gpt-4o-mini", api_key="SUA_API_KEY")

# Exemplo de criação dos agentes utilizando o objeto instanciado
content_creator_agent = Agent(
    config=agents_config['content_creator_agent'],
    tools=[SerperDevTool(), WebsiteSearchTool()],
    llm=llm_model  # Agora llm_model é um objeto, não uma string
)

quality_assurance_agent = Agent(
    config=agents_config['quality_assurance_agent'],
    llm=llm_model
)


ModuleNotFoundError: No module named 'crewai.llms'

In [7]:
# type: ignore
# Creating Agents
market_news_monitor_agent = Agent(
    config=agents_config['market_news_monitor_agent'],
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    llm=groq_llm,
)

data_analyst_agent = Agent(
    config=agents_config['data_analyst_agent'],
    tools=[SerperDevTool(), WebsiteSearchTool()],
    llm=groq_llm,
)

content_creator_agent = Agent(
    config=agents_config['content_creator_agent'],
    tools=[SerperDevTool(), WebsiteSearchTool()],
    llm=llm_model
)

quality_assurance_agent = Agent(
    config=agents_config['quality_assurance_agent'],
    llm=llm_model
)

# Creating Tasks
monitor_financial_news_task = Task(
    config=tasks_config['monitor_financial_news'],
    agent=market_news_monitor_agent
)

analyze_market_data_task = Task(
    config=tasks_config['analyze_market_data'],
    agent=data_analyst_agent
)

create_content_task = Task(
    config=tasks_config['create_content'],
    agent=content_creator_agent,
    context=[monitor_financial_news_task, analyze_market_data_task]
)

quality_assurance_task = Task(
    config=tasks_config['quality_assurance'],
    agent=quality_assurance_agent,
    output_pydantic=ContentOutput
)

# Creating Crew
content_creation_crew = Crew(
    agents=[
        market_news_monitor_agent,
        data_analyst_agent,
        content_creator_agent,
        quality_assurance_agent
    ],
    tasks=[
        monitor_financial_news_task,
        analyze_market_data_task,
        create_content_task,
        quality_assurance_task
    ],
    verbose=True
)

2025-02-01 21:06:05,511 - 9828 - llm.py-llm:293 - ERROR: Failed to get supported params: argument of type 'NoneType' is not iterable


LLM value is a string

Provider List: https://docs.litellm.ai/docs/providers



2025-02-01 21:06:06,048 - 9828 - llm.py-llm:293 - ERROR: Failed to get supported params: argument of type 'NoneType' is not iterable


LLM value is a string

Provider List: https://docs.litellm.ai/docs/providers



2025-02-01 21:06:06,394 - 9828 - llm.py-llm:293 - ERROR: Failed to get supported params: argument of type 'NoneType' is not iterable
2025-02-01 21:06:06,397 - 9828 - llm.py-llm:293 - ERROR: Failed to get supported params: argument of type 'NoneType' is not iterable


LLM value is already an LLM object
LLM value is already an LLM object

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



## Kicking off the Crew

In [36]:
result = content_creation_crew.kickoff(inputs={
  'subject': 'Inflação nos EUA e o impacto no mercado de ações em 2024'
})

2025-01-30 16:35:42,077 - 26228 - llm.py-llm:293 - ERROR: Failed to get supported params: argument of type 'NoneType' is not iterable
2025-01-30 16:35:42,079 - 26228 - llm.py-llm:293 - ERROR: Failed to get supported params: argument of type 'NoneType' is not iterable


2025-01-30 16:35:42,083 - 26228 - llm.py-llm:293 - ERROR: Failed to get supported params: argument of type 'NoneType' is not iterable
2025-01-30 16:35:42,087 - 26228 - llm.py-llm:277 - ERROR: LiteLLM call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=llama-3.3-70b-versatile
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
2025-01-30 16:35:42,091 - 26228 - llm.py-llm:277 - ERROR: LiteLLM call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=llama-3.3-70b-versatile
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
2025-01-30 16:35:42,095 - 26228 - llm.py-llm:277 - ERROR: LiteLLM call failed: litellm.BadReques


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

# Agent: Analista Chefe de Mercado
## Task: Monitore e analise as Ãºltimas notÃ­cias e atualizaÃ§Ãµes relacionadas aos mercados financeiros, com um foco especÃ­fico em Inflação nos EUA e o impacto no mercado de ações em 2024. Identifique e resuma os itens de notÃ­cias mais relevantes e impactantes que possam influenciar as tendÃªncias do mercado ou as decisÃµes dos investidores. Utilize APIs de notÃ­cias financeiras e ferramentas de dados de mercado em tempo real para coletar informaÃ§Ãµes atualizadas. Foque em detectar tendÃªncias, mudanÃ§as regulatÃ³rias ou indicadores econÃ´micos significativos que estejam diretamente relacionados a Inflação nos EUA e o impacto no mercado de ações em 2024.


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: h

2025-01-30 16:35:42,231 - 26228 - llm.py-llm:277 - ERROR: LiteLLM call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=llama-3.3-70b-versatile
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
2025-01-30 16:35:42,234 - 26228 - llm.py-llm:277 - ERROR: LiteLLM call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=llama-3.3-70b-versatile
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
2025-01-30 16:35:42,238 - 26228 - llm.py-llm:277 - ERROR: LiteLLM call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=llama-3.3-70b-versatile
 Pass mod


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



2025-01-30 16:35:42,262 - 26228 - llm.py-llm:277 - ERROR: LiteLLM call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=llama-3.3-70b-versatile
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
2025-01-30 16:35:42,265 - 26228 - llm.py-llm:277 - ERROR: LiteLLM call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=llama-3.3-70b-versatile
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
2025-01-30 16:35:42,269 - 26228 - llm.py-llm:277 - ERROR: LiteLLM call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=llama-3.3-70b-versatile
 Pass mod

... (718 outputs left out)


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



2025-01-30 16:36:16,012 - 26228 - llm.py-llm:277 - ERROR: LiteLLM call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=llama-3.3-70b-versatile
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
2025-01-30 16:36:16,016 - 26228 - llm.py-llm:277 - ERROR: LiteLLM call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=llama-3.3-70b-versatile
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
2025-01-30 16:36:16,019 - 26228 - llm.py-llm:277 - ERROR: LiteLLM call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=llama-3.3-70b-versatile
 Pass mod


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



KeyboardInterrupt: 